In [1]:
# ============================================================
# 04_visualizaciones_mapas.ipynb
# Mapas interactivos de valor del suelo — Bogotá D.C.
# Fuente: UAECD Catastro Distrital
# ============================================================

import geopandas as gpd
import pandas as pd
import folium
from folium.plugins import Fullscreen, MiniMap, HeatMap
import os
import warnings
warnings.filterwarnings("ignore")

RUTA_PROCESSED = r"C:\Users\kenpa\OneDrive\Desktop\FORMACION\colombia-mercado-inmobiliario\data\processed"
RUTA_REPORTS   = r"C:\Users\kenpa\OneDrive\Desktop\FORMACION\colombia-mercado-inmobiliario\reports"
os.makedirs(RUTA_REPORTS, exist_ok=True)

print("✓ Librerías cargadas")
print(f"  folium : {folium.__version__}")

✓ Librerías cargadas
  folium : 0.15.0


In [ ]:
# ── Cargar dataset con localidades ───────────────────────────
gdf = gpd.read_file(
    os.path.join(RUTA_PROCESSED, "bogota_valor_localidades.gpkg"),
    engine="pyogrio"
)
print(f"✅ Dataset cargado: {len(gdf):,} registros")

# ── Construir resumen por localidad para el mapa ─────────────
resumen_loc = (
    gdf.groupby(["localidad", "año"])["valor_ref_m2"]
    .agg(mediana="median", media="mean", manzanas="count")
    .reset_index()
)

# Pivot para tener 2021 y 2024 en columnas
pivot = resumen_loc.pivot_table(
    index="localidad", columns="año", values="mediana"
).reset_index()
pivot.columns = ["localidad", "med_2021", "med_2022", "med_2024"]
pivot["var_pct"] = ((pivot["med_2024"] - pivot["med_2021"]) / pivot["med_2021"] * 100).round(1)

# ── Unir con geometrías de localidades ───────────────────────
# Usar el centroide del conjunto de manzanas por localidad
geo_loc = (
    gdf[gdf["año"] == 2024]
    .dissolve(by="localidad")
    .reset_index()[["localidad", "geometry"]]
)
geo_loc = geo_loc.merge(pivot, on="localidad", how="left")
geo_loc = geo_loc.to_crs("EPSG:4326")

print(f"✅ Capa de localidades lista: {len(geo_loc)} polígonos")
print(f"📊 Columnas: {geo_loc.columns.tolist()}")

✅ Dataset cargado: 125,003 registros
✅ Capa de localidades lista: 20 polígonos
 📊 Columnas: ['localidad', 'geometry', 'med_2021', 'med_2022', 'med_2024', 'var_pct']


In [4]:
# ── Centro del mapa — Bogotá ──────────────────────────────────
centro = [4.6097, -74.0817]

m1 = folium.Map(
    location=centro,
    zoom_start=11,
    tiles="CartoDB positron"
)

# ── Escala de color por valor mediano 2024 ────────────────────
import branca.colormap as cm

valores = geo_loc["med_2024"].dropna()
colormap = cm.LinearColormap(
    colors=["#FFF5EB", "#FD8D3C", "#D94701"],
    vmin=valores.min(),
    vmax=valores.max(),
    caption="Valor de referencia mediano (COP/m²) — 2024"
)
colormap.add_to(m1)

# ── Capa Choropleth ───────────────────────────────────────────
def estilo_valor(feature):
    val = feature["properties"].get("med_2024")
    if val is None or pd.isna(val):
        return {"fillColor": "#cccccc", "color": "white",
                "weight": 1.5, "fillOpacity": 0.5}
    return {
        "fillColor": colormap(val),
        "color":     "white",
        "weight":    1.5,
        "fillOpacity": 0.75
    }

def highlight(feature):
    return {"weight": 3, "color": "#333333", "fillOpacity": 0.9}

# ── Tooltip con info detallada ────────────────────────────────
tooltip = folium.GeoJsonTooltip(
    fields=["localidad", "med_2021", "med_2022", "med_2024", "var_pct"],
    aliases=["📍 Localidad", "💰 Valor 2021 ($/m²)",
             "💰 Valor 2022 ($/m²)", "💰 Valor 2024 ($/m²)", "📈 Variación 2021→2024 (%)"],
    localize=True,
    sticky=True,
    style="""
        background-color: white;
        border: 1px solid #ccc;
        border-radius: 6px;
        padding: 8px;
        font-family: Arial, sans-serif;
        font-size: 13px;
        box-shadow: 2px 2px 6px rgba(0,0,0,0.2);
    """
)

folium.GeoJson(
    geo_loc.__geo_interface__,
    name="Valor referencia 2024",
    style_function=estilo_valor,
    highlight_function=highlight,
    tooltip=tooltip
).add_to(m1)

# ── Plugins ───────────────────────────────────────────────────
Fullscreen(position="topright").add_to(m1)
MiniMap(toggle_display=True).add_to(m1)
folium.LayerControl().add_to(m1)

# ── Título ────────────────────────────────────────────────────
titulo = """
<div style="position: fixed; top: 15px; left: 55px; z-index: 1000;
     background: white; padding: 10px 16px; border-radius: 8px;
     border: 1px solid #ccc; box-shadow: 2px 2px 8px rgba(0,0,0,0.2);
     font-family: Arial; font-size: 14px; max-width: 320px;">
    <b>🏙️ Valor del Suelo — Bogotá D.C.</b><br>
    <span style="font-size:12px; color:#555;">
        Valor de referencia comercial por m² | Vigencia 2024<br>
        Fuente: UAECD Catastro Distrital
    </span>
</div>
"""
m1.get_root().html.add_child(folium.Element(titulo))

# ── Guardar ───────────────────────────────────────────────────
ruta_m1 = os.path.join(RUTA_REPORTS, "mapa_01_valor_2024.html")
m1.save(ruta_m1)
print(f" Mapa 1 guardado: {ruta_m1}")
print("  Ábrelo en tu navegador para verlo interactivo")

 Mapa 1 guardado: C:\Users\kenpa\OneDrive\Desktop\FORMACION\colombia-mercado-inmobiliario\reports\mapa_01_valor_2024.html
  Ábrelo en tu navegador para verlo interactivo


In [5]:
m2 = folium.Map(
    location=centro,
    zoom_start=11,
    tiles="CartoDB positron"
)

# ── Escala divergente: rojo=baja, blanco=neutro, verde=sube ──
colormap_var = cm.LinearColormap(
    colors=["#D73027", "#FFFFBF", "#1A9850"],
    vmin=-20,
    vmax=40,
    caption="Variación valor de referencia (%) — 2021 → 2024"
)
colormap_var.add_to(m2)

def estilo_variacion(feature):
    val = feature["properties"].get("var_pct")
    if val is None or pd.isna(val):
        return {"fillColor": "#cccccc", "color": "white",
                "weight": 1.5, "fillOpacity": 0.5}
    return {
        "fillColor": colormap_var(val),
        "color":     "white",
        "weight":    1.5,
        "fillOpacity": 0.80
    }

# ── Popup con mini-análisis por localidad ────────────────────
def crear_popup(row):
    var = row.get("var_pct", 0) or 0
    emoji = "🟢" if var >= 15 else "🟡" if var >= 0 else "🔴"
    tendencia = "Alta valorización" if var >= 15 else \
                "Valorización moderada" if var >= 5 else \
                "Estable" if var >= 0 else "Desvalorización"
    html = f"""
    <div style="font-family:Arial; font-size:13px; min-width:220px; padding:4px">
        <h4 style="margin:0 0 8px; color:#1A5276">
            📍 {row.get('localidad','N/A').title()}
        </h4>
        <table style="width:100%; border-collapse:collapse">
            <tr><td style="color:#555">Valor 2021</td>
                <td style="text-align:right; font-weight:bold">
                    ${row.get('med_2021',0)/1e6:.2f}M/m²</td></tr>
            <tr><td style="color:#555">Valor 2022</td>
                <td style="text-align:right; font-weight:bold">
                    ${row.get('med_2022',0)/1e6:.2f}M/m²</td></tr>
            <tr><td style="color:#555">Valor 2024</td>
                <td style="text-align:right; font-weight:bold">
                    ${row.get('med_2024',0)/1e6:.2f}M/m²</td></tr>
            <tr style="border-top:1px solid #eee">
                <td style="color:#555">Variación</td>
                <td style="text-align:right; font-weight:bold;
                    color:{'#1A9850' if var>=0 else '#D73027'}">
                    {var:+.1f}%</td></tr>
        </table>
        <div style="margin-top:8px; padding:5px 8px; border-radius:4px;
            background:{'#EAF7EA' if var>=0 else '#FDEDEC'};
            font-size:12px">
            {emoji} {tendencia}
        </div>
    </div>
    """
    return folium.Popup(html, max_width=280)

# ── Agregar polígonos con popup individual ────────────────────
for _, row in geo_loc.iterrows():
    if row.geometry is None:
        continue
    folium.GeoJson(
        row.geometry.__geo_interface__,
        style_function=lambda x, r=row: {
            "fillColor": colormap_var(r["var_pct"])
                         if pd.notna(r["var_pct"]) else "#ccc",
            "color": "white",
            "weight": 1.5,
            "fillOpacity": 0.80
        },
        highlight_function=highlight,
        tooltip=f"{row['localidad'].title()} — {row['var_pct']:+.1f}%"
                if pd.notna(row.get("var_pct")) else row["localidad"],
        popup=crear_popup(row)
    ).add_to(m2)

# ── Título ────────────────────────────────────────────────────
titulo2 = """
<div style="position: fixed; top: 15px; left: 55px; z-index: 1000;
     background: white; padding: 10px 16px; border-radius: 8px;
     border: 1px solid #ccc; box-shadow: 2px 2px 8px rgba(0,0,0,0.2);
     font-family: Arial; font-size: 14px; max-width: 340px;">
    <b>📈 Valorización del Suelo — Bogotá D.C.</b><br>
    <span style="font-size:12px; color:#555;">
        Variación del valor comercial por m² | 2021 → 2024<br>
        🟢 Verde = mayor valorización &nbsp;|&nbsp; 🔴 Rojo = caída<br>
        Fuente: UAECD Catastro Distrital
    </span>
</div>
"""
m2.get_root().html.add_child(folium.Element(titulo2))

Fullscreen(position="topright").add_to(m2)
MiniMap(toggle_display=True).add_to(m2)

ruta_m2 = os.path.join(RUTA_REPORTS, "mapa_02_valorizacion_2021_2024.html")
m2.save(ruta_m2)
print(f"✓ Mapa 2 guardado: {ruta_m2}")

✓ Mapa 2 guardado: C:\Users\kenpa\OneDrive\Desktop\FORMACION\colombia-mercado-inmobiliario\reports\mapa_02_valorizacion_2021_2024.html


In [6]:
# Este mapa usa los ~42.000 puntos individuales — muy impactante visualmente

m3 = folium.Map(
    location=centro,
    zoom_start=12,
    tiles="CartoDB dark_matter"   # fondo oscuro resalta el heatmap
)

# Calcular centroides de cada manzana para el heatmap
gdf_2024 = gdf[gdf["año"] == 2024].copy()
gdf_2024 = gdf_2024.to_crs("EPSG:4326")
gdf_2024["centroid"] = gdf_2024.geometry.centroid

# Preparar datos para HeatMap: [lat, lon, peso]
heat_data = [
    [row.centroid.y, row.centroid.x, row.valor_ref_m2 / 1e6]
    for _, row in gdf_2024.iterrows()
    if row.centroid is not None and pd.notna(row.valor_ref_m2)
]

HeatMap(
    heat_data,
    radius=10,
    blur=8,
    min_opacity=0.3,
    gradient={0.2: "#313695", 0.4: "#4575B4",
              0.6: "#FEE090", 0.8: "#F46D43", 1.0: "#A50026"}
).add_to(m3)

titulo3 = """
<div style="position: fixed; top: 15px; left: 55px; z-index: 1000;
     background: rgba(0,0,0,0.75); padding: 10px 16px; border-radius: 8px;
     font-family: Arial; font-size: 14px; max-width: 320px; color: white;">
    <b>🔥 Concentración del Valor del Suelo</b><br>
    <span style="font-size:12px; color:#ccc;">
        Heatmap por manzana | 42.000+ registros | 2024<br>
        Rojo intenso = mayor valor comercial/m²<br>
        Fuente: UAECD Catastro Distrital
    </span>
</div>
"""
m3.get_root().html.add_child(folium.Element(titulo3))
Fullscreen(position="topright").add_to(m3)

ruta_m3 = os.path.join(RUTA_REPORTS, "mapa_03_heatmap_manzanas.html")
m3.save(ruta_m3)
print(f"✓ Mapa 3 guardado: {ruta_m3}")
print("\n" + "="*50)
print("MAPAS GENERADOS")
print("="*50)
for nombre in ["mapa_01_valor_2024.html",
               "mapa_02_valorizacion_2021_2024.html",
               "mapa_03_heatmap_manzanas.html"]:
    ruta = os.path.join(RUTA_REPORTS, nombre)
    size = os.path.getsize(ruta) / 1024
    print(f"  ✓ {nombre} — {size:.0f} KB")

✓ Mapa 3 guardado: C:\Users\kenpa\OneDrive\Desktop\FORMACION\colombia-mercado-inmobiliario\reports\mapa_03_heatmap_manzanas.html

MAPAS GENERADOS
  ✓ mapa_01_valor_2024.html — 100532 KB
  ✓ mapa_02_valorizacion_2021_2024.html — 100595 KB
  ✓ mapa_03_heatmap_manzanas.html — 1919 KB
